# Evaluation Architecture Notes

This notebook is a text-style walkthrough of the evaluation layer.

The main idea is:
- the universe stores the shared dataset
- each construction stores fixed weights and in-sample metrics
- backtesting and Monte Carlo are done later
- evaluation results are attached back into the stored construction
- no rolling rebalance and no repeated re-optimization are used here

## 1. What is being evaluated?

Each construction is interpreted as a **fixed-weight portfolio**.

That means:
- weights are built once on a construction window
- those weights are held fixed afterward
- the backtest measures how those fixed weights behaved in a later period
- the Monte Carlo engine simulates future outcomes for those same fixed weights

In [ ]:
from pathlib import Path

from portafolios import Universe
from portafolios.data.local_loader import local_loader
from portafolios.constructores.equal_weight import EqualWeightConstructor
from portafolios.eval import Backtester, MonteCarloEngine

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / "data_clean_stock_data.csv"

print("CSV exists:", CSV_PATH.exists())

CSV exists: True


## 2. Build one shared universe

The universe contains:
- prices
- returns
- tickers
- metadata

Here we load a full year of data, but later we will say that the construction window only used the first half of the year.

In [3]:
u = Universe(
    tickers=["AAPL", "MSFT", "AMZN"],
    start="2020-01-01",
    end="2020-12-31",
    loader=local_loader,
    loader_kwargs={"path": CSV_PATH, "prefer_adj_close": True},
    freq="B",
).preparar_datos()

u.prices.head()

,AAPL,MSFT,AMZN
Date,,,
2020-01-02,72.776606,153.428246,94.900497
2020-01-03,72.771729,152.683705,94.309998
2020-01-06,72.621654,151.872308,95.184502
2020-01-07,72.849231,152.416407,95.694504
2020-01-08,73.706271,153.495104,95.550003


In [4]:
print("Universe metadata:", u.metadata)
print("Prices shape:", u.prices.shape)
print("Returns shape:", u.returns.shape)

Universe metadata: {'source': 'callable_dataframe', 'frequency': 'B'}
Prices shape: (261, 3)
Returns shape: (260, 3)


## 3. Build and store constructions

Each construction is saved inside the same universe.

Important detail:
- `construction_start` and `construction_end` identify the estimation window
- this is what lets the evaluation layer validate that the backtest begins later

In [ ]:
u.construir(

    Naive(),

    label="equal_weight",

    ann_factor=252,

    construction_start="2020-01-01",

    construction_end="2020-06-30",

)

u.construir(

    Naive(),

    label="naive",

    ann_factor=252,

    construction_start="2020-01-01",

    construction_end="2020-06-30",

)

u.list_constructions()

['equal_weight', 'naive']

In [6]:
stored = u.get_construction("equal_weight")
print("Construction name:", stored.name)
print("Method:", stored.method)
print("Construction window:", stored.construction_start, "->", stored.construction_end)
print("In-sample metrics:", stored.metrics_insample)
stored.weights

Construction name: equal_weight
Method: equal_weight
Construction window: 2020-01-01 00:00:00 -> 2020-06-30 00:00:00
In-sample metrics: {'n_selected': 3, 'expected_return': 0.5341655857966859, 'volatility': 0.29135818256004353, 'sharpe_m': 1.833363940916965}


AAPL    0.333333
AMZN    0.333333
MSFT    0.333333
dtype: float64

## 4. Fixed-weight backtesting

The backtester:
- retrieves the saved construction
- takes its fixed weights
- checks that the backtest starts after the construction window
- computes realized portfolio returns over the chosen period
- computes cumulative returns and summary metrics
- attaches the result back into the saved construction

In [7]:
bt = Backtester(u, "equal_weight", ann_factor=252)
bt_result = bt.run_and_attach(
    start_date="2020-07-01",
    end_date="2020-12-31",
    notes="Fixed-weight evaluation after the construction window.",
)

bt_result.summary_metrics

{'n_periods': 132,
 'total_return': 0.25107373974278224,
 'annualized_return': 0.533634662800281,
 'annualized_volatility': 0.27589200479867493,
 'sharpe_ratio': 1.9342157565953648,
 'max_drawdown': -0.1656216701706721}

In [8]:
u.get_construction("equal_weight").backtest_result.summary_metrics

{'n_periods': 132,
 'total_return': 0.25107373974278224,
 'annualized_return': 0.533634662800281,
 'annualized_volatility': 0.27589200479867493,
 'sharpe_ratio': 1.9342157565953648,
 'max_drawdown': -0.1656216701706721}

## 5. Monte Carlo simulation

The Monte Carlo engine:
- retrieves the same fixed weights
- estimates mean vector and covariance from the universe returns
- simulates future asset returns with a multivariate normal model
- converts them into portfolio paths
- stores the result back inside the same construction

In [9]:
mc = MonteCarloEngine(u, "equal_weight", seed=42)
mc_result = mc.run_and_attach(
    horizon=20,
    n_simulations=500,
    initial_value=1.0,
    notes="Simple multivariate normal Monte Carlo.",
)

mc_result.summary_metrics

{'mean_terminal_value': 1.0367677630261896,
 'median_terminal_value': 1.0368938153391698,
 'std_terminal_value': 0.08568992336175575,
 'min_terminal_value': 0.7944711347937535,
 'max_terminal_value': 1.3019151524350563,
 'prob_loss': 0.354,
 'mean_terminal_return': 0.03676776302618954}

In [10]:
u.get_construction("equal_weight").mc_result.summary_metrics

{'mean_terminal_value': 1.0367677630261896,
 'median_terminal_value': 1.0368938153391698,
 'std_terminal_value': 0.08568992336175575,
 'min_terminal_value': 0.7944711347937535,
 'max_terminal_value': 1.3019151524350563,
 'prob_loss': 0.354,
 'mean_terminal_return': 0.03676776302618954}

## 6. Evaluate all stored constructions

The same evaluation layer can run across all saved constructions in the universe.

In [11]:
all_bt = Backtester.run_all(
    u,
    start_date="2020-07-01",
    end_date="2020-12-31",
    ann_factor=252,
    attach=True,
)

{name: result.summary_metrics for name, result in all_bt.items()}

{'equal_weight': {'n_periods': 132,
  'total_return': 0.25107373974278224,
  'annualized_return': 0.533634662800281,
  'annualized_volatility': 0.27589200479867493,
  'sharpe_ratio': 1.9342157565953648,
  'max_drawdown': -0.1656216701706721},
 'naive': {'n_periods': 132,
  'total_return': 0.25107373974278224,
  'annualized_return': 0.533634662800281,
  'annualized_volatility': 0.27589200479867493,
  'sharpe_ratio': 1.9342157565953648,
  'max_drawdown': -0.1656216701706721}}

In [12]:
all_mc = MonteCarloEngine.run_all(
    u,
    horizon=20,
    n_simulations=250,
    initial_value=1.0,
    attach=True,
    seed=123,
)

{name: result.summary_metrics for name, result in all_mc.items()}

{'equal_weight': {'mean_terminal_value': 1.032270916339112,
  'median_terminal_value': 1.0360110228727248,
  'std_terminal_value': 0.0855124156722261,
  'min_terminal_value': 0.8169219459690407,
  'max_terminal_value': 1.2365508328664323,
  'prob_loss': 0.384,
  'mean_terminal_return': 0.0322709163391119},
 'naive': {'mean_terminal_value': 1.0343935242828197,
  'median_terminal_value': 1.0351353136414914,
  'std_terminal_value': 0.07758846346022027,
  'min_terminal_value': 0.8165383009654167,
  'max_terminal_value': 1.2550591752460922,
  'prob_loss': 0.312,
  'mean_terminal_return': 0.03439352428281978}}

## 7. Final interpretation

The architecture is now separated in three layers:

1. Data layer
   - loads and standardizes prices and returns

2. Construction layer
   - creates fixed weights and in-sample metrics
   - stores each construction in the universe

3. Evaluation layer
   - backtests fixed weights after the construction window
   - runs Monte Carlo on the saved fixed weights
   - attaches both results back to the corresponding construction